# Train IRL Gaslight 5M

This notebook trains a ~4.94M-parameter defensive-language classifier and curated-response retriever. It does not train an unrestricted manipulation or harassment chatbot.

Before running: **Runtime -> Change runtime type -> T4 GPU**.

In [ ]:
from pathlib import Path
import subprocess

repo = Path('/content/irl-gaslight')
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Dev2Creator/irl-gaslight.git', str(repo)], check=True)
%cd /content/irl-gaslight
!python -m pip install -q -r ml/requirements-colab.txt

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
assert torch.cuda.is_available(), 'Enable a T4 GPU from Runtime -> Change runtime type.'

## Train and export
The script builds group-safe train/validation/test splits, trains the tokenizer and multi-task model, restores the best checkpoint, evaluates it, builds retrieval embeddings, and exports float and INT8 ONNX files.

In [ ]:
!python ml/train_5m.py --epochs 14 --batch-size 64 --patience 3 --output artifacts/irl-gaslight-5m

In [ ]:
import json
from pathlib import Path

artifact_dir = Path('artifacts/irl-gaslight-5m')
metrics = json.loads((artifact_dir / 'metrics.json').read_text())
metrics

## Try a defensive pattern check

In [ ]:
!python ml/infer_5m.py --model-dir artifacts/irl-gaslight-5m --text "That never happened. You imagined it."

## Download the artifact bundle
Keep the metrics and tokenizer with the model. Review performance and false positives before making it the default CLI engine.

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive('/content/irl-gaslight-5m', 'zip', 'artifacts/irl-gaslight-5m')
files.download(archive)

### Optional: save to Google Drive

In [ ]:
# Uncomment to keep a copy in Drive.
# from google.colab import drive
# drive.mount('/content/drive')
# shutil.copy2('/content/irl-gaslight-5m.zip', '/content/drive/MyDrive/irl-gaslight-5m.zip')